# Week1_ex1+ : Torque vs. Rotation Angle Sweep

## Background

In `Week1_ex1.ipynb`, the torque on a current-carrying coil was evaluated at a single fixed rotation angle (45 deg), producing a torque of **28.29 uN*m**. This notebook extends that setup into a parametric sweep to check whether the torque follows the theoretically expected **sin(theta)** pattern as the coil rotates through a full revolution.

## Purpose

- Register the coil rotation angle (`rot_angle`) as a design variable and sweep it from 0 deg to 345 deg in 15 deg steps (24 points total).
- Extract the torque (`Torque1.Torque`) at each angle and plot torque vs. rotation angle.
- Compare the resulting curve against the sin(theta) prediction from a simple magnetostatic torque model.

## Method

1. Build the same coil + permanent magnet geometry as `Week1_ex1.ipynb`, but assign `rot_angle` as a variable and rotate the coil (and its current-carrying cross-sections) about the Z axis by that variable.
2. Replace the single-point `analyze()` call with a `LinearStep` parametric sweep (`AngleSweep`) over `rot_angle`, saving fields at every step.
3. Retrieve `Torque1.Torque` for all 24 variations with `get_solution_data()`, convert to a DataFrame, and plot.

## Note on units

`get_solution_data().data_real()` returns values in SI base units (N*m), not the uN*m unit shown in the AEDT GUI. The torque values extracted here are multiplied by 1e6 to convert to uN*m before plotting, so they can be compared directly against the 28.29 uN*m reference value from the 45 deg fixed-angle run.

## Result summary (see `Week1_ex1_angle_sweep_analysis.ipynb` for the full investigation)

The torque broadly follows a sin(theta)-like trend, but a sharp, symmetric dip appears exactly at 90 deg and 270 deg. A separate diagnostic notebook (mesh convergence check + Top-view B-field comparison) traces this to the coil-magnet geometry passing through a symmetric alignment at those angles, rather than to numerical/meshing error.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import time

In [ ]:
## Initialize AEDT Desktop session and create Maxwell 3D project/design ##

# Start AEDT session (specify version and student license flag)
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# Disable autosave to prevent popup interruptions during scripted runs
DT.disable_autosave()

# Solution type: Magnetostatic
sol_type = "Magnetostatic"

# Create Maxwell 3D design object
# On first creation, a new project and design are generated with default names
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)
# Reference to odesign, used for AEDT's built-in recording-style API calls
oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##

# Project name (change freely as needed)
proj_name = "Week1_ex1_angle_sweep"

# Compute save directory path relative to this notebook's location
# (portable across machines; no hardcoded personal path)
dir = os.getcwd() + f"\\{proj_name}"
print(dir)

# Create directory if it doesn't exist yet (safe to re-run; won't error if already there)
os.makedirs(dir, exist_ok=True)

# Design name
desi_name = "Week1_ex1_angle_sweep"


In [ ]:
## Save project and apply design name ##

# Reference to the project object containing this design
proj = M3D.oproject

# Save project with the target file name (directory created above)
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)

# Rename design to match the target design name
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()

In [ ]:
## Create geometry ##

# Sketch a circle and sweep it around an axis to form a ring-shaped coil

origin = [0, 5, 0]
coil = M3D.modeler.create_circle(orientation="XY", origin=origin, radius=0.5, num_sides=12, is_covered=True, name="Coil", material=None, non_model=False)

M3D.modeler.sweep_around_axis(assignment=coil, axis="X", sweep_angle=360, draft_angle=0, number_of_segments=30)

M3D.assign_material(assignment=coil, material="copper")

# Create the bar-shaped permanent magnet
origin = [-3, -0.5, -0.5]
sizes = [6, 1, 1]
magnet = M3D.modeler.create_box(origin, sizes, name="Magnet", material="NdFe35")


In [ ]:
## Check magnet material (NdFe35) magnetization direction / coercivity ##

NdFe35 = M3D.materials.exists_material(material="NdFe35")

display(NdFe35.get_magnetic_coercivity())

# # Use this if the magnetization direction differs from expected (uncomment if needed)
# NdFe35.set_magnetic_coercivity(value='-890000A_per_meter', x="1", y="0", z="0")

In [ ]:
## Create coil terminal cross-sections for current excitation ##

coil_section = []
coil = [coil]   # Wrap in a list to allow loop processing
for c in coil : 
    M3D.modeler.section(assignment=c, plane="XY", create_new=True, section_cross_object=False)
    coil_section.append( M3D.modeler.sheet_objects[-1] )

M3D.modeler.split(assignment=coil_section, plane="ZX", sides="NegativeOnly", tool=None, split_crossing_objs=False, delete_invalid_objs=True)

# Assign current excitation to each coil cross-section

coil_terminal = []
for s in coil_section :
    coil_terminal.append( M3D.assign_current(assignment=s, amplitude="100A", phase='0deg', solid=False, swap_direction=False, name=None) )


In [ ]:
## Assign virtual torque boundary condition to the magnet ##

M3D.assign_torque(assignment=magnet, coordinate_system='Global', is_positive=True, is_virtual=True, axis='Z', torque_name="Torque1")


In [ ]:
## Rotate coil to set relative angle with the magnet ##

# Register rotation angle as a design variable so it can be swept later
M3D["rot_angle"] = "45deg"

M3D.modeler.rotate(assignment=coil, axis="Z", angle="rot_angle")
M3D.modeler.rotate(assignment=coil_section, axis="Z", angle="rot_angle")

In [ ]:
## Create simulation region (surrounding air/vacuum domain) ##

region = M3D.modeler.create_region(pad_value=100, pad_type='Percentage Offset', name='Region')

In [ ]:
## Create and inspect analysis setup ##

# Create analysis setup object
my_setup = M3D.create_setup(name="Setup1")

# Check available setup properties for the current solution type (stored as a dict)
display(my_setup.props)

In [ ]:
# Modify desired properties from the dict above (maximum number of adaptive passes)

my_setup.props['MaximumPasses'] = 10


In [ ]:
## Set up and run parametric sweep over rotation angle ##

angle_sweep = M3D.parametrics.add(variable="rot_angle", start_point="0deg", end_point="345deg",
                                   step="15deg", variation_type="LinearStep", name="AngleSweep")

angle_sweep.props["ProdOptiSetupDataV2"]["SaveFields"] = True
angle_sweep.update()

angle_sweep.analyze()

In [ ]:
## Extract torque data across all angles ##

data = M3D.post.get_solution_data(
    expressions="Torque1.Torque",
    setup_sweep_name="Setup1 : LastAdaptive",
    primary_sweep_variable="rot_angle",
    variations={"rot_angle": ["All"]},
)

angles = np.array(data.primary_sweep_values, dtype=float)

# NOTE: get_solution_data().data_real() returns values in SI base units (N*m),
# not the uN*m unit shown in the AEDT GUI. Multiply by 1e6 to convert to uN*m
# so this matches the 28.29 uN*m reference value from the 45 deg fixed-angle run.
torque = np.array(data.data_real("Torque1.Torque"), dtype=float) * 1e6

df = pd.DataFrame({"angle_deg": angles, "Torque_uNm": torque})
df = df.sort_values("angle_deg").reset_index(drop=True)
df

In [ ]:
## Plot torque vs rotation angle ##
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))
plt.plot(df["angle_deg"], df["Torque_uNm"], marker="o")
plt.xlabel("Coil rotation angle [deg]")
plt.ylabel("Torque [uN*m]")
plt.title("Torque vs Rotation Angle")
plt.grid(True)
plt.tight_layout()
plt.savefig("Images/Week1_ex1_angle_sweep_plot.png", dpi=150)
plt.show()

df.to_csv("Week1_ex1_angle_sweep_result.csv", index=False)

In [ ]:
# Save final results
M3D.save_project()